In [ ]:
"""Full-corpus native Decider inference on 2×T4; private model from Kaggle input."""
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import zipfile

ROOT = Path("/tmp/innovation-group7")
OUT = Path("/tmp/humanoid-production")
SAVED = Path("/kaggle/working")
SAVED.mkdir(parents=True, exist_ok=True)
PINNED_COMMIT = "REPLACE_GIT_SHA"

def check(cmd, **kw):
    print("RUNNING:", " ".join(map(str, cmd)), flush=True)
    subprocess.run(cmd, check=True, **kw)

check(["git", "clone", "--no-checkout", "https://github.com/roccoangelella/Innovation-management-group-7.git", str(ROOT)])
check(["git", "-C", str(ROOT), "checkout", "--detach", PINNED_COMMIT])
summary = ROOT / "humanoid_patents_en_summary.json"
source = ROOT / "humanoid_patents_en.zip"
assert summary.is_file() and source.is_file(), "Final corpus and translation summary must be committed"
translated = json.loads(summary.read_text())
assert translated["incomplete"] == 0 and translated["completed"] == 237538, "Full corpus not validated"
print("CORPUS_READY", translated["completed"], translated["final_archive_sha256"], flush=True)

input_root = Path("/kaggle/input")
print("KAGGLE_INPUT_DIRS", sorted(str(p) for p in input_root.iterdir()) if input_root.is_dir() else [], flush=True)
checkpoint = Path("/tmp/pplx-private-model")
manifest_paths = list(input_root.rglob("snapshot_manifest.json")) if input_root.is_dir() else []
print("FOUND_PRIVATE_MODEL_MANIFESTS", len(manifest_paths), flush=True)
if len(manifest_paths) == 1:
    manifest_path = manifest_paths[0]
    snapshot_files = json.loads(manifest_path.read_text(encoding="utf-8"))
    for entry in snapshot_files:
        destination = (checkpoint / entry["path"]).resolve()
        if not destination.is_relative_to(checkpoint.resolve()):
            raise ValueError("Unsafe checkpoint filename")
        source_file = manifest_path.parent / entry["flat_name"]
        if not source_file.is_file():
            raise FileNotFoundError(source_file)
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source_file, destination)
else:
    raise FileNotFoundError("Expected exactly one mounted private model snapshot_manifest.json; inspect KAGGLE_INPUT_DIRS")
assert (checkpoint / "source" / "src").exists(), "Model source not found"
print("MODEL_STAGED", flush=True)

# When running a later version, attach the private checkpoint dataset with
# humanoid-checkpoint.zip. This run otherwise starts from empty journals.
resume = list(Path("/kaggle/input").rglob("humanoid-checkpoint.zip"))
if len(resume) > 1:
    raise RuntimeError("Multiple checkpoint artifacts attached; use exactly one")
if resume:
    OUT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(resume[0]) as archive:
        for entry in archive.infolist():
            dest = (OUT / entry.filename).resolve()
            if not dest.is_relative_to(OUT.resolve()):
                raise ValueError("Unexpected checkpoint archive path")
        archive.extractall(OUT)
    print("RESUMED_FROM_CHECKPOINT", resume[0], flush=True)

try:
    check([
        sys.executable, "-u", str(ROOT / "kaggle/production_run.py"),
        "--input", str(source),
        "--translation-summary", str(summary),
        "--production-dir", str(OUT),
        "--checkpoint", str(checkpoint),
        "--skip-download",
        "--session-minutes", "300",
    ], cwd=ROOT)
finally:
    # Always save the audit/progress state, even on a failed model run.
    with zipfile.ZipFile(SAVED / "humanoid-checkpoint.zip", "w", zipfile.ZIP_DEFLATED, compresslevel=5) as archive:
        for name in [
            "production_manifest.json", "worker_0/run_metadata.json",
            "worker_1/run_metadata.json", "worker_0/predictions.jsonl",
            "worker_1/predictions.jsonl", "session_summary.json",
            "production_summary.json",
        ]:
            f = OUT / name
            if f.is_file():
                archive.write(f, arcname=name)
    for name in ("predictions.csv", "filtered_patents.zip", "production_summary.json", "session_summary.json"):
        f = OUT / name
        if f.is_file():
            shutil.copy2(f, SAVED / name)
    print("CHECKPOINT_SAVED", (SAVED / "humanoid-checkpoint.zip").stat().st_size, flush=True)
